In [2]:
# =============================================================================
# Synthetic Phase 3 Clinical-Trial Dataset Generator
# ==================================================
# DISCLAIMER: the output is 100 % synthetic. No real trial, drug, sponsor or patient
# is represented. It must NOT be used to predict real-world trial outcomes or to
# support clinical, regulatory, investment or portfolio decisions.
#
# Purpose: reproduce the structure of a reference Phase 3 dataset (same 17 columns, similar
# category mix, rates and effect sizes) WITHOUT its clipped "capped value" spikes at 10,
# 0.99, 0 and 1 year. Variables are bounded by construction (Beta / log-normal / Poisson).
# The only clip is enrollment >= 40, which affects ~1 row; it is counted and reported.
#
# Success model: additive logistic regression + Bernoulli noise. The intercept is solved so
# the EXPECTED success rate is TARGET_SUCCESS; the realised rate differs by sampling noise.
#
# Usage (terminal):  python synthetic_dataset_generator.py --seed 973 --out data/synthetic_clinical_trial_phase3.csv
# Usage (notebook):  main([])        # empty list stops argparse reading the kernel's arguments
# =============================================================================

import numpy as np
import pandas as pd
from scipy.optimize import brentq   # or use a manual binary search
from scipy.stats import beta as beta_dist

# ------------------------------------------------------------
# 0. Settings
# ------------------------------------------------------------
N_ROWS         = 42_000
SEED           = 973
TARGET_SUCCESS = 0.55      # we want 55 % successes
rng            = np.random.default_rng(SEED)

# ------------------------------------------------------------
# 1. Helper to draw a bounded beta with a chosen mean/SD
# ------------------------------------------------------------
def beta_from_mean_sd(mean, sd, size, low=0.0, high=1.0):
    m = (mean - low) / (high - low)
    s = sd / (high - low)
    common = m * (1 - m) / (s ** 2) - 1
    a = m * common
    b = (1 - m) * common
    return low + (high - low) * rng.beta(a, b, size=size)

# ------------------------------------------------------------
# 2. Generate all *predictor* columns
# ------------------------------------------------------------
areas = ["oncology","cns","infectious_disease","cardiovascular",
         "rare_disease","immunology","metabolic","vaccine","respiratory"]
area_probs = np.array([10872,4975,4277,4167,3812,3784,3671,3461,2981])
area_probs = area_probs/area_probs.sum()

endpoints = ["clinical","surrogate","hard_survival"]
endpoint_probs = np.array([16771,12717,12512])/42000

sponsors = ["small_biotech","large_pharma","mid_pharma","academic"]
sponsor_probs = np.array([14436,14155,8472,4937])/42000

therapeutic_area = rng.choice(areas, size=N_ROWS, p=area_probs)
endpoint_type    = rng.choice(endpoints, size=N_ROWS, p=endpoint_probs)
sponsor_type     = rng.choice(sponsors, size=N_ROWS, p=sponsor_probs)

uses_biomarker     = rng.binomial(1, 0.417, N_ROWS)
novel_mechanism    = rng.binomial(1, 0.418, N_ROWS)
randomized         = rng.binomial(1, 0.880, N_ROWS)
double_blind       = rng.binomial(1, 0.720, N_ROWS)
active_comparator  = rng.binomial(1, 0.495, N_ROWS)
orphan_designation = rng.binomial(1, 0.181, N_ROWS)

phase2_effect_size = beta_from_mean_sd(mean=5.0, sd=1.95, size=N_ROWS,
                                       low=0, high=10)

latent      = 0.40 * (phase2_effect_size - 5.0) / 1.95 + rng.normal(0, 1, N_ROWS)
percentile  = pd.Series(latent).rank(pct=True).to_numpy()
m, s        = 0.69, 0.20
common      = m * (1 - m) / s**2 - 1
a_pv, b_pv  = m * common, (1 - m) * common
phase2_pvalue_strength = 10 * beta_dist.ppf(percentile*0.9999+0.00005,
                                            a_pv, b_pv)

statistical_power = beta_from_mean_sd(mean=0.82, sd=0.10, size=N_ROWS,
                                      low=0.40, high=0.995)

log_enroll = rng.normal(np.log(480), 0.62, N_ROWS) + 0.8*(statistical_power-0.82)
enrollment = np.round(np.exp(log_enroll)).astype(int)
enrollment = np.maximum(enrollment, 40)

prior_approvals_target = rng.poisson(1.2, N_ROWS)

years_in_development = beta_from_mean_sd(mean=7.0, sd=2.5, size=N_ROWS,
                                         low=1.0, high=17.0)

# ------------------------------------------------------------
# 3. Fixed (non‑intercept) part of the log‑odds
# ------------------------------------------------------------
area_effect = {"oncology": -0.30, "cns": -0.20, "rare_disease": -0.08,
               "respiratory": 0.08, "immunology": 0.10,
               "cardiovascular": 0.15, "metabolic": 0.28,
               "infectious_disease": 0.38, "vaccine": 0.45}
endpoint_effect = {"hard_survival": -0.30, "clinical": 0.0,
                   "surrogate": 0.28}
sponsor_effect = {"academic": -0.20, "small_biotech": -0.12,
                  "mid_pharma": 0.05, "large_pharma": 0.28}

fixed_log_odds = (
    0.33 * (phase2_effect_size - 5.0) / 1.95
    + 0.22 * (phase2_pvalue_strength - 6.9) / 2.0
    + 0.20 * (statistical_power - 0.82) / 0.10
    + 0.30 * uses_biomarker
    - 0.20 * novel_mechanism
    + 0.10 * (prior_approvals_target - 1.2)
    + 0.08 * randomized
    + 0.04 * double_blind
    - 0.02 * active_comparator
    + 0.05 * orphan_designation
    + 0.02 * (np.log(enrollment) - np.log(500))
    + pd.Series(therapeutic_area).map(area_effect).to_numpy()
    + pd.Series(endpoint_type).map(endpoint_effect).to_numpy()
    + pd.Series(sponsor_type).map(sponsor_effect).to_numpy()
)

# ------------------------------------------------------------
# 4. Helper that returns the *mean* success proportion for a given intercept
# ------------------------------------------------------------
def mean_success(intercept):
    """Return the expected overall success rate if we use `intercept` as the
    constant term in the logistic model."""
    log_odds = intercept + fixed_log_odds
    prob     = 1 / (1 + np.exp(-log_odds))
    return prob.mean()

# ------------------------------------------------------------
# 5. Find the intercept that gives exactly 55 %
# ------------------------------------------------------------
# The function `mean_success(intercept) - TARGET_SUCCESS` is monotone in the
# intercept, so a simple brentq (bisection) works.
intercept_solution = brentq(
    lambda b: mean_success(b) - TARGET_SUCCESS,
    a=-2.0,          # lower bound – definitely below target
    b= 2.0,          # upper bound – definitely above target
    xtol=1e-6
)

print(f"Calibrated intercept → {intercept_solution:.5f}")
print(f"Resulting overall success rate → {mean_success(intercept_solution):.5%}")

# ------------------------------------------------------------
# 6. Generate the final label using the calibrated intercept
# ------------------------------------------------------------
log_odds   = intercept_solution + fixed_log_odds
prob_success = 1 / (1 + np.exp(-log_odds))
trial_success = rng.binomial(1, prob_success)   # no post‑hoc flipping needed

# ------------------------------------------------------------
# 7. Assemble the dataframe (identical column order to the original)
# ------------------------------------------------------------
synthetic = pd.DataFrame({
    "trial_id":               [f"SYN{i:06d}" for i in range(N_ROWS)],
    "therapeutic_area":       therapeutic_area,
    "uses_biomarker":         uses_biomarker,
    "phase2_effect_size":     np.round(phase2_effect_size, 2),
    "phase2_pvalue_strength": np.round(phase2_pvalue_strength, 2),
    "endpoint_type":          endpoint_type,
    "enrollment":             enrollment,
    "statistical_power":      np.round(statistical_power, 3),
    "sponsor_type":           sponsor_type,
    "novel_mechanism":        novel_mechanism,
    "prior_approvals_target": prior_approvals_target,
    "randomized":             randomized,
    "double_blind":           double_blind,
    "active_comparator":      active_comparator,
    "years_in_development":   np.round(years_in_development, 1),
    "orphan_designation":     orphan_designation,
    "trial_success":          trial_success,
})

synthetic.to_csv("synthetic_clinical_trial_phase3.csv",
                 index=False)

print("\n✅ Finished!")
print(f"Shape: {synthetic.shape}")
print(f"Overall success rate: {synthetic['trial_success'].mean():.3%}")

# ------------------------------------------------------------
# 8. Quick sanity checks
# ------------------------------------------------------------
if __name__ == "__main__":
    print("\n--- Success balance ---")
    print(synthetic["trial_success"].value_counts(normalize=True).round(3))

    print("\n--- Numeric summary (sample) ---")
    num_cols = ["phase2_effect_size","phase2_pvalue_strength","enrollment",
                "statistical_power","prior_approvals_target",
                "years_in_development"]
    print(synthetic[num_cols].describe().round(2))

    print("\n--- Success rate by key categories (showing a few) ---")
    for col in ["therapeutic_area","endpoint_type","sponsor_type",
                "uses_biomarker","novel_mechanism"]:
        print(synthetic.groupby(col)["trial_success"].mean().round(3))
        print()

Calibrated intercept → 0.02650
Resulting overall success rate → 54.99999%

✅ Finished!
Shape: (42000, 17)
Overall success rate: 55.036%

--- Success balance ---
trial_success
1    0.55
0    0.45
Name: proportion, dtype: float64

--- Numeric summary (sample) ---
       phase2_effect_size  phase2_pvalue_strength  enrollment  \
count            42000.00                42000.00    42000.00   
mean                 5.00                    6.90      582.18   
std                  1.96                    2.00      403.07   
min                  0.08                    0.35       45.00   
25%                  3.53                    5.56      315.00   
50%                  5.01                    7.21      481.00   
75%                  6.47                    8.53      729.00   
max                  9.97                   10.00     6897.00   

       statistical_power  prior_approvals_target  years_in_development  
count           42000.00                42000.00              42000.00  
mean  